# 01 — Preprocessing

**Thesis:** An Explainable Early-Warning System for Predicting At-Risk Graduate Trainees in Banking Cybersecurity Onboarding Using XGBoost and SHAP: A Low-Resource Sub-Saharan African Validation

This notebook loads the raw survey data (28-item instrument), handles missing values, derives SDT composite scores, encodes categorical variables, and produces stratified train/test splits for the modelling pipeline.

**Input:** `data/dummy_gmt_survey_data.csv` (replace with real survey export post-HuSSREC clearance)

**Output:** `data/train_processed.csv`, `data/test_processed.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

RANDOM_SEED = 42
DATA_PATH = "data/dummy_gmt_survey_data.csv"
TARGET_COL = "at_risk"

## 1. Load and inspect data

In [2]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (350, 28)


,age,gender,highest_education,years_prior_work_experience,bank_name,it_related_degree,prior_cyber_certification,prior_cyber_training,self_rated_it_proficiency,training_attendance_rate,...,sdt_competence_3,sdt_relatedness_1,sdt_relatedness_2,sdt_relatedness_3,perceived_content_difficulty,perceived_workload,perceived_supervisor_support,self_rated_confidence,self_reported_quiz_score,at_risk
0,21,Male,Bachelor's,3,Stanbic,No,No,No,2,84.6,...,5,6,6,4,3,3,3.0,2,82.8,0
1,27,Male,Professional Cert,3,GCB,Yes,No,Yes,4,89.0,...,5,4,4,3,3,2,2.0,3,80.7,1
2,26,Female,Professional Cert,2,Ecobank,No,No,No,4,43.7,...,1,4,5,2,4,4,1.0,2,NaN,1
3,24,Female,Master's,1,GCB,Yes,No,No,4,82.0,...,4,4,3,3,3,4,3.0,4,69.7,1
4,24,Female,Bachelor's,2,Ecobank,Yes,No,No,1,87.0,...,5,6,6,7,2,2,4.0,4,99.8,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   age                           350 non-null    int64  
 1   gender                        350 non-null    object 
 2   highest_education             350 non-null    object 
 3   years_prior_work_experience   350 non-null    int64  
 4   bank_name                     350 non-null    object 
 5   it_related_degree             350 non-null    object 
 6   prior_cyber_certification     350 non-null    object 
 7   prior_cyber_training          350 non-null    object 
 8   self_rated_it_proficiency     350 non-null    int64  
 9   training_attendance_rate      350 non-null    float64
 10  modules_completed_pct         350 non-null    float64
 11  avg_time_per_module_minutes   343 non-null    float64
 12  engagement_score              350 non-null    int64  
 13  sdt_a

In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,350.0,NaN,NaN,NaN,25.008571,2.50671,21.0,23.0,25.0,27.0,29.0
gender,350,2,Male,187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
highest_education,350,3,Bachelor's,238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_prior_work_experience,350.0,NaN,NaN,NaN,1.431429,1.099432,0.0,0.0,1.0,2.0,3.0
bank_name,350,6,GCB,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
it_related_degree,350,2,No,233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prior_cyber_certification,350,2,No,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prior_cyber_training,350,2,No,238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
self_rated_it_proficiency,350.0,NaN,NaN,NaN,3.108571,1.448123,1.0,2.0,3.0,4.0,5.0
training_attendance_rate,350.0,NaN,NaN,NaN,70.802571,12.162798,36.3,63.025,70.2,78.775,100.0


## 2. Missing value check and imputation

Numeric survey items with missing values are imputed using the column median. For the real dataset, consider whether listwise deletion or a more sophisticated imputation strategy (e.g., MICE) is more appropriate depending on the missingness rate and pattern (MCAR/MAR/MNAR).

In [5]:
missing_summary = df.isna().sum()
missing_cols = missing_summary[missing_summary > 0]
print("Columns with missing values:")
print(missing_cols)

Columns with missing values:
avg_time_per_module_minutes     7
perceived_supervisor_support    7
self_reported_quiz_score        7
dtype: int64


In [6]:
if len(missing_cols) > 0:
    num_imputer = SimpleImputer(strategy="median")
    df[missing_cols.index] = num_imputer.fit_transform(df[missing_cols.index])

assert df.isna().sum().sum() == 0, "Missing values remain after imputation"
print("Missing values handled. Total remaining NaNs:", df.isna().sum().sum())

Missing values handled. Total remaining NaNs: 0


## 3. Derive SDT composite scores

Self-Determination Theory (SDT) is operationalised via three subscales (autonomy, competence, relatedness), each measured with 3 items on a 1-7 Likert scale. Composite scores are computed as the mean of each subscale's items.

In [7]:
df["sdt_autonomy_score"] = df[["sdt_autonomy_1", "sdt_autonomy_2", "sdt_autonomy_3"]].mean(axis=1)
df["sdt_competence_score"] = df[["sdt_competence_1", "sdt_competence_2", "sdt_competence_3"]].mean(axis=1)
df["sdt_relatedness_score"] = df[["sdt_relatedness_1", "sdt_relatedness_2", "sdt_relatedness_3"]].mean(axis=1)

df[["sdt_autonomy_score", "sdt_competence_score", "sdt_relatedness_score"]].describe()

,sdt_autonomy_score,sdt_competence_score,sdt_relatedness_score
count,350.000000,350.000000,350.000000
mean,4.006667,4.078095,4.033333
std,1.112984,1.235523,0.987908
min,1.000000,1.000000,1.000000
25%,3.333333,3.333333,3.333333
50%,4.000000,4.000000,4.000000
75%,4.666667,5.000000,4.666667
max,7.000000,7.000000,7.000000


## 4. Encode categorical variables

Nominal categorical variables (demographics, prior IT exposure flags) are one-hot encoded with the first category dropped to avoid the dummy variable trap. Ordinal Likert-scale items are already numeric and left as-is.

In [8]:
categorical_cols = [
    "gender", "highest_education", "bank_name",
    "it_related_degree", "prior_cyber_certification", "prior_cyber_training",
]

encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(df[categorical_cols])
encoded_cols = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)

df_processed = pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)
print("Processed shape:", df_processed.shape)
df_processed.head()

Processed shape: (350, 36)


,age,years_prior_work_experience,self_rated_it_proficiency,training_attendance_rate,modules_completed_pct,avg_time_per_module_minutes,engagement_score,sdt_autonomy_1,sdt_autonomy_2,sdt_autonomy_3,...,highest_education_Master's,highest_education_Professional Cert,bank_name_AccessBank,bank_name_Ecobank,bank_name_GCB,bank_name_Other,bank_name_Stanbic,it_related_degree_Yes,prior_cyber_certification_Yes,prior_cyber_training_Yes
0,21,3,2,84.6,39.9,55.9,3,5,6,3,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,27,3,4,89.0,69.9,35.8,3,4,5,4,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,26,2,4,43.7,40.2,44.7,1,1,4,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,24,1,4,82.0,62.4,40.0,3,7,4,5,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,24,2,1,87.0,100.0,36.9,4,6,4,5,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


**Note (sklearn version compatibility):** `OneHotEncoder(sparse_output=...)` requires scikit-learn >= 1.2. If running on an older sklearn (< 1.2), replace `sparse_output` with `sparse=False`.

## 5. Stratified train/test split

An 80/20 split, stratified on the target (`at_risk`), preserves the class imbalance ratio across both sets.

In [9]:
X = df_processed.drop(columns=[TARGET_COL])
y = df_processed[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape: (280, 35) | Test shape: (70, 35)

Train class balance:
at_risk
0    0.660714
1    0.339286
Name: proportion, dtype: float64

Test class balance:
at_risk
0    0.657143
1    0.342857
Name: proportion, dtype: float64


## 6. Save processed splits

In [10]:
train_df = X_train.copy()
train_df[TARGET_COL] = y_train
test_df = X_test.copy()
test_df[TARGET_COL] = y_test

train_df.to_csv("data/train_processed.csv", index=False)
test_df.to_csv("data/test_processed.csv", index=False)

print("Saved data/train_processed.csv and data/test_processed.csv")

Saved data/train_processed.csv and data/test_processed.csv


## Smoke test checklist
- [ ] Notebook runs top-to-bottom with no errors
- [ ] `data/train_processed.csv` and `data/test_processed.csv` are created
- [ ] No missing values remain
- [ ] Train/test class balance proportions are similar

Proceed to **02_modelling_xgboost.ipynb**.